# KanKyouKen SDK - Basic Usage

This notebook demonstrates basic usage of the KanKyouKen Python SDK for querying event data.

## Setup

First, install the SDK if you haven't already:

```bash
cd sdk
pip install -e ".[pandas]"
```

In [ ]:
from kankyouken import KanKyouKenClient
from datetime import datetime, timedelta
import os

## Initialize Client

Create a client instance. You can either:
1. Pass URL and token directly
2. Use environment variables (KANKYOUKEN_URL, KANKYOUKEN_TOKEN)

In [ ]:
# Option 1: Explicit parameters
client = KanKyouKenClient(
    url="http://localhost:54321",
    token="your-jwt-token-here"
)

# Option 2: From environment (recommended)
# client = KanKyouKenClient()

print(f"Connected to: {client.url}")

## Query Events by Study

Basic query to get all events for a study:

In [ ]:
# Replace with your actual study ID
STUDY_ID = "your-study-id-here"

response = client.query_events(study_id=STUDY_ID, limit=10)

print(f"Total events: {response.pagination.total}")
print(f"Returned: {response.pagination.returned}")
print(f"\nFirst few events:")

for event in response.events[:5]:
    print(f"  {event.ts}: {event.event_type} (participant: {event.participant_id})")

## Filter Events

Apply filters to narrow down results:

In [ ]:
# Get only login events from the last 7 days
response = client.query_events(
    study_id=STUDY_ID,
    event_type="login",
    date_from=datetime.now() - timedelta(days=7),
    limit=50
)

print(f"Login events in last 7 days: {response.pagination.total}")
print(f"\nEvent types: {set(e.event_type for e in response.events)}")

## Convert to DataFrame

Convert events to pandas DataFrame for analysis:

In [ ]:
response = client.query_events(study_id=STUDY_ID, limit=100)
df = response.to_dataframe()

print("DataFrame shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

## Analyze Events

Use pandas to analyze the data:

In [ ]:
# Count events by type
event_counts = df['event_type'].value_counts()
print("Event counts by type:")
print(event_counts)

# Count events by participant
participant_counts = df['participant_id'].value_counts()
print("\nEvents per participant:")
print(participant_counts.describe())

## Iterate Through All Events

Use `iter_events()` to automatically handle pagination:

In [ ]:
total_events = 0
event_types = set()

# Iterate through all events in batches of 100
for page in client.iter_events(study_id=STUDY_ID, page_size=100):
    total_events += len(page.events)
    event_types.update(e.event_type for e in page.events)
    print(f"Fetched {len(page.events)} events (total so far: {total_events})")

print(f"\nTotal events processed: {total_events}")
print(f"Unique event types: {event_types}")

## Query by Project

Get events from all studies in a project:

In [ ]:
# Replace with your actual project ID
PROJECT_ID = "your-project-id-here"

response = client.query_events(project_id=PROJECT_ID, limit=50)

print(f"Total events across all studies: {response.pagination.total}")
print(f"Studies represented: {len(set(e.study_id for e in response.events))}")

# Group by study
df = response.to_dataframe()
study_counts = df['study_id'].value_counts()
print("\nEvents per study:")
print(study_counts)

## Advanced: Custom Analysis

Example of more complex analysis:

In [ ]:
# Get all events and convert to DataFrame
all_events = []
for page in client.iter_events(study_id=STUDY_ID, page_size=500):
    all_events.extend(page.events)

# Analyze event payload data
print(f"Total events: {len(all_events)}")

# Look at payload fields
payload_keys = set()
for event in all_events:
    if event.payload:
        payload_keys.update(event.payload.keys())

print(f"Payload fields found: {payload_keys}")

# Time-based analysis
df_all = response.to_dataframe()
df_all['hour'] = df_all['ts'].dt.hour
hourly_activity = df_all['hour'].value_counts().sort_index()

print("\nActivity by hour of day:")
print(hourly_activity)

## Next Steps

- See `02_event_analysis.ipynb` for event-specific analysis patterns
- See `03_ml_integration.ipynb` for machine learning examples
- Check the SDK documentation for all available options